In [ ]:
# Run if kaggle
import os

base_path = '/kaggle/working'
project_name = 'FreeCell-Solver'
project_root = os.path.join(base_path, project_name)

if not os.path.exists(project_root):
    os.chdir(base_path)
    !git clone https://github.com/nthday-jpg/FreeCell-Solver.git
    print("Repository cloned successfully!")
else:
    os.chdir(project_root)
    !git pull
    print("Repository updated successfully!")
%cd {project_root}

from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [ ]:
from __future__ import annotations

import multiprocessing as mp
import os
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from statistics import mean
from time import perf_counter
from collections.abc import Callable
from typing import TypedDict
from joblib import Parallel, delayed


from freecell.core.state import GameState
from freecell.solvers.IDS import IDSSolver
from freecell.solvers.Astar import AstarSolver

SCHEDULERS: dict[str, Callable[[int], int]] = {}

class RunRow(TypedDict):
    seed: int
    solved: bool
    move_count: int
    expanded_nodes: int
    solver_elapsed: float
    wall_elapsed: float
    astar_solved: bool
    astar_move_count: int | None
    match_astar_len: bool | None


class TaggedRunRow(RunRow):
    scheduler: str


class SummaryRow(TypedDict):
    scheduler: str
    solved: int
    total: int
    solve_rate: float
    avg_wall_s: float
    avg_nodes: float
    avg_moves_when_solved: float
    match_astar_count: int
    match_astar_comparable: int
    match_astar_rate: float | None

def run_ids_once(
    seed: int,
    scheduler_name: str,
    *,
    max_depth: int,
    max_expansions: int,
    astar_max_expansions: int = 300_000,
) -> RunRow:
    scheduler = SCHEDULERS[scheduler_name]
    initial = GameState.initial(seed=seed).to_packed()
    solver = IDSSolver(
        max_depth=max_depth,
        max_expansions=max_expansions,
        depth_limit_scheduler=scheduler,
    )

    started = perf_counter()
    result = solver.solve(initial)
    wall_time = perf_counter() - started

    astar = AstarSolver(max_expansions=astar_max_expansions)
    astar_result = astar.solve(initial)

    astar_move_count: int | None = astar_result.move_count if astar_result.solved else None
    match_astar_len: bool | None = None
    if result.solved and astar_move_count is not None:
        match_astar_len = result.move_count == astar_move_count

    return {
        "seed": seed,
        "solved": result.solved,
        "move_count": result.move_count,
        "expanded_nodes": result.expanded_nodes,
        "solver_elapsed": result.elapsed_seconds,
        "wall_elapsed": wall_time,
        "astar_solved": astar_result.solved,
        "astar_move_count": astar_move_count,
        "match_astar_len": match_astar_len,
    }

def tune_depth_schedulers(
    seeds: range,
    *,
    max_depth: int = 80,
    max_expansions: int = 300_000,
    astar_max_expansions: int = 300_000,
    workers: int | None = None,
) -> tuple[list[TaggedRunRow], list[SummaryRow], str]:
    all_rows: list[TaggedRunRow] = []
    summary: list[SummaryRow] = []

    max_workers = workers if workers is not None else (os.cpu_count() or 1)

    for name in SCHEDULERS.keys():
        rows: list[RunRow] = Parallel(n_jobs=max_workers, backend="loky")(
            delayed(run_ids_once)(
                seed,
                name,
                max_depth=max_depth,
                max_expansions=max_expansions,
                astar_max_expansions=astar_max_expansions,
            )
            for seed in seeds
        )

        for row in rows:
            all_rows.append(
                {
                    "scheduler": name,
                    "seed": row["seed"],
                    "solved": row["solved"],
                    "move_count": row["move_count"],
                    "expanded_nodes": row["expanded_nodes"],
                    "solver_elapsed": row["solver_elapsed"],
                    "wall_elapsed": row["wall_elapsed"],
                    "astar_solved": row["astar_solved"],
                    "astar_move_count": row["astar_move_count"],
                    "match_astar_len": row["match_astar_len"],
                }
            )

        solved_rows = [r for r in rows if r["solved"]]
        solved_count = len(solved_rows)
        total = len(rows)

        comparable_rows = [r for r in rows if r["match_astar_len"] is not None]
        match_count = sum(1 for r in comparable_rows if r["match_astar_len"] is True)
        comparable_total = len(comparable_rows)

        summary.append(
            {
                "scheduler": name,
                "solved": solved_count,
                "total": total,
                "solve_rate": solved_count / total if total else 0.0,
                "avg_wall_s": mean(r["wall_elapsed"] for r in rows) if rows else 0.0,
                "avg_nodes": mean(r["expanded_nodes"] for r in rows) if rows else 0.0,
                "avg_moves_when_solved": (
                    mean(r["move_count"] for r in solved_rows) if solved_rows else 0.0
                ),
                "match_astar_count": match_count,
                "match_astar_comparable": comparable_total,
                "match_astar_rate": (match_count / comparable_total) if comparable_total else None,
            }
        )

    return all_rows, summary, "joblib-loky"

In [ ]:
def step580(step: int) -> int:
    return 80 + step * 5

def step585(step: int) -> int:
    return 85 + step * 5


SCHEDULERS["step580"] = step580
SCHEDULERS["step585"] = step585


In [ ]:
# ---- Run tuning ----
SEEDS = range(1, 11)
MAX_WORKERS = min(os.cpu_count() or 1, len(SEEDS))
print(f"Using up to {MAX_WORKERS} workers")

all_rows, summary, executor_kind = tune_depth_schedulers(
    SEEDS,
    max_depth=100,
    max_expansions=300_000,
    astar_max_expansions=300_000,
    workers=MAX_WORKERS,
    )

print(f"Executor kind: {executor_kind}")
if executor_kind != "joblib-loky":
    print("Warning: process-based parallelism is not active; CPU scaling may be limited.")

summary_sorted = sorted(summary, key=lambda r: (-r["solve_rate"], r["avg_wall_s"]))
print("Depth-scheduler tuning summary")
print("=" * 110)
print(
    f"{'scheduler':<14} {'solved':>7} {'rate':>8} {'avg_wall_s':>12} {'avg_nodes':>12} {'A*_match':>10} {'A*_rate':>9}"
)
for r in summary_sorted:
    match_rate = r["match_astar_rate"]
    match_rate_text = f"{match_rate:.2%}" if match_rate is not None else "n/a"
    print(
        f"{r['scheduler']:<14} "
        f"{str(r['solved']) + '/' + str(r['total']):>7} "
        f"{r['solve_rate']:>8.2%} "
        f"{r['avg_wall_s']:>12.3f} "
        f"{r['avg_nodes']:>12.0f} "
        f"{str(r['match_astar_count']) + '/' + str(r['match_astar_comparable']):>10} "
        f"{match_rate_text:>9}"
    )

# Optional: inspect mismatches vs A* reference length
[r for r in all_rows if r['scheduler'] == 'linear' and r['match_astar_len'] is False][:5]